# Stage 6/7a — AfriBERTa Embedding, Near-Duplicate Detection, BERTopic, AfriSenti

Reads `ngx.tweets_resolved` from BigQuery, embeds text with AfriBERTa, flags
near-duplicates, fits BERTopic once on the full corpus, runs AfriSenti sentiment,
and writes results back to BigQuery for stage 7b (feature computation).

**Run order matters.** Execute cells top to bottom. Each expensive step checkpoints
to GCS, so a disconnect only costs you the current batch, not the whole run.

**Before running:** confirm GPU + High-RAM below. Use L4, not T4 or A100, for this
workload.

In [16]:
!gcloud auth revoke --all

ERROR: (gcloud.auth.revoke) Cannot revoke GCE-provided credentials.


In [2]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated.")

Authenticated.


In [3]:
!nvidia-smi
import psutil
print(f"{psutil.virtual_memory().total/1e9:.1f} GB RAM available")

Fri Sep 18 22:07:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
PROJECT = "ngx-discourse-2026"
DATASET = "ngx"
BUCKET  = "ngx-discourse-2026-raw"
TWEETS_RESOLVED   = f"{DATASET}.tweets_resolved"
TWEET_EMBEDDINGS  = f"{DATASET}.tweet_embeddings"
TWEET_TOPICS      = f"{DATASET}.tweet_topics"
TWEET_SENTIMENT   = f"{DATASET}.tweet_sentiment"
TWEET_DUPLICATES  = f"{DATASET}.tweet_duplicates"
EMBED_CHECKPOINT_PREFIX = f"gs://{BUCKET}/derived/embeddings/"
MODEL_NAME = "castorini/afriberta_large"
BATCH_SIZE = 128
DUP_SIM_THRESHOLD = 0.97   # cosine similarity above this = near-duplicateprint("Config loaded.")

In [12]:
!pip install -q transformers sentence-transformers bertopic faiss-cpu \
    google-cloud-bigquery google-cloud-storage db-dtypes tqdm

import torch, transformers, bertopic
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("bertopic:", bertopic.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.3 MB/s eta 0:00:00
torch: 2.11.0+cu128
transformers: 5.16.1
bertopic: 0.17.4
CUDA available: True
Device: NVIDIA L4


In [17]:
from google.cloud import bigquery
client = bigquery.Client(project="ngx-discourse-2026")

query = '''
SELECT tweet_id, trading_day, created_at, text, text_status,
       author_hash, retweet_of_id, reply_to_hash
FROM `ngx-discourse-2026.ngx.tweets_resolved`
WHERE in_analysis_window
  AND text_status != 'truncated'
  AND text IS NOT NULL
  AND LENGTH(text) > 0
'''

df = client.query(query).result().to_dataframe()
print(f"Loaded {len(df):,} tweets with usable text")
print(df['text_status'].value_counts())

Loaded 150,607 tweets with usable text
text_status
complete     116881
recovered     33726
Name: count, dtype: int64


In [19]:
import time
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"Model loaded on {device}")

@torch.no_grad()
def embed_batch(texts, max_length=128):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length,
                     return_tensors="pt").to(device)
    out = model(**enc)
    cls = out.last_hidden_state[:, 0, :]  # [CLS] token
    return cls.cpu().numpy()

# Timing test on a 5,000-tweet sample to extrapolate the full run
sample = df.sample(min(5000, len(df)), random_state=42).reset_index(drop=True)
t0 = time.time()
for i in range(0, len(sample), BATCH_SIZE):
    batch_texts = sample['text'].iloc[i:i+BATCH_SIZE].tolist()
    _ = embed_batch(batch_texts)
elapsed = time.time() - t0
rate = len(sample) / elapsed
eta_full = len(df) / rate

print(f"{len(sample)} tweets embedded in {elapsed:.1f}s -> {rate:.1f} tweets/sec")
print(f"Estimated time for full corpus ({len(df):,} tweets): {eta_full/60:.1f} minutes")

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 1.55MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

[transformers] XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  503MB            

model.safetensors: downloading bytes:           |  0.00B            

5000 tweets embedded in 12.1s -> 414.8 tweets/sec
Estimated time for full corpus (150,607 tweets): 6.1 minutes


In [20]:
import numpy as np
import pandas as pd
from google.cloud import storage
import io, time

storage_client = storage.Client(project=PROJECT)
bucket = storage_client.bucket(BUCKET)

CHECKPOINT_EVERY = 20   # batches
existing_ids = set()

# Resume support: check which tweet_ids are already embedded
blobs = list(storage_client.list_blobs(BUCKET, prefix="derived/embeddings/"))
if blobs:
    print(f"Found {len(blobs)} existing checkpoint file(s), checking for already-embedded IDs...")
    for b in blobs:
        if b.name.endswith(".parquet"):
            buf = io.BytesIO(b.download_as_bytes())
            existing_ids.update(pd.read_parquet(buf, columns=['tweet_id'])['tweet_id'].tolist())
    print(f"{len(existing_ids):,} tweets already embedded, will be skipped")

todo = df[~df['tweet_id'].isin(existing_ids)].reset_index(drop=True)
print(f"{len(todo):,} tweets remaining to embed")

t_start = time.time()
buffer_ids, buffer_vecs = [], []
checkpoint_num = len(blobs)

for batch_num, i in enumerate(range(0, len(todo), BATCH_SIZE)):
    batch = todo.iloc[i:i+BATCH_SIZE]
    vecs = embed_batch(batch['text'].tolist())
    buffer_ids.extend(batch['tweet_id'].tolist())
    buffer_vecs.append(vecs)

    if (batch_num + 1) % CHECKPOINT_EVERY == 0 or i + BATCH_SIZE >= len(todo):
        all_vecs = np.vstack(buffer_vecs)
        out_df = pd.DataFrame({
            'tweet_id': buffer_ids,
            'embedding': list(all_vecs)
        })
        checkpoint_num += 1
        blob_name = f"derived/embeddings/part-{checkpoint_num:05d}.parquet"
        buf = io.BytesIO()
        out_df.to_parquet(buf, index=False)
        buf.seek(0)
        bucket.blob(blob_name).upload_from_file(buf, content_type="application/octet-stream")

        elapsed = time.time() - t_start
        done = i + len(batch)
        rate = done / elapsed if elapsed > 0 else 0
        remaining = (len(todo) - done) / rate if rate > 0 else 0
        print(f"Checkpoint {checkpoint_num}: {done:,}/{len(todo):,} embedded, rate {rate:.1f} tweets/sec, {remaining/60:.1f} min remaining")

        buffer_ids, buffer_vecs = [], []

print("Embedding complete.")

150,607 tweets remaining to embed
Checkpoint 1: 2,560/150,607 embedded, rate 259.1 tweets/sec, 9.5 min remaining
Checkpoint 2: 5,120/150,607 embedded, rate 259.4 tweets/sec, 9.3 min remaining
Checkpoint 3: 7,680/150,607 embedded, rate 257.8 tweets/sec, 9.2 min remaining
Checkpoint 4: 10,240/150,607 embedded, rate 257.2 tweets/sec, 9.1 min remaining
Checkpoint 5: 12,800/150,607 embedded, rate 256.2 tweets/sec, 9.0 min remaining
Checkpoint 6: 15,360/150,607 embedded, rate 255.6 tweets/sec, 8.8 min remaining
Checkpoint 7: 17,920/150,607 embedded, rate 255.1 tweets/sec, 8.7 min remaining
Checkpoint 8: 20,480/150,607 embedded, rate 254.6 tweets/sec, 8.5 min remaining
Checkpoint 9: 23,040/150,607 embedded, rate 253.8 tweets/sec, 8.4 min remaining
Checkpoint 10: 25,600/150,607 embedded, rate 253.1 tweets/sec, 8.2 min remaining
Checkpoint 11: 28,160/150,607 embedded, rate 252.5 tweets/sec, 8.1 min remaining
Checkpoint 12: 30,720/150,607 embedded, rate 251.2 tweets/sec, 8.0 min remaining
Checkp

In [22]:
# Reload every checkpoint and consolidate
blobs = list(storage_client.list_blobs(BUCKET, prefix="derived/embeddings/"))
frames = []
for b in blobs:
    if b.name.endswith(".parquet"):
        buf = io.BytesIO(b.download_as_bytes())
        frames.append(pd.read_parquet(buf))

embeddings_df = pd.concat(frames, ignore_index=True).drop_duplicates(subset='tweet_id')
print(f"Total embeddings: {len(embeddings_df):,}")
print(f"Embedding dimension: {len(embeddings_df['embedding'].iloc[0])}")

# Write to BigQuery as an array of float64
embeddings_df['embedding'] = embeddings_df['embedding'].apply(lambda v: [float(x) for x in v])

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
load_job = client.load_table_from_dataframe(
    embeddings_df, f"{PROJECT}.{TWEET_EMBEDDINGS}", job_config=job_config)
load_job.result()
print(f"Loaded {len(embeddings_df):,} rows to {TWEET_EMBEDDINGS}")

Total embeddings: 150,607
Embedding dimension: 768
Loaded 150,607 rows to ngx.tweet_embeddings


## Near-duplicate detection
Uses a local FAISS index over the embeddings just computed. Flags tweets whose nearestneighbour (excluding themselves) exceeds the similarity threshold — catches templated spam("Cost of Financial Illiteracy", GTBank scam blocks) before it inflates f7/f10/f11.

In [23]:
import faiss

vecs = np.stack(embeddings_df['embedding'].apply(np.array).values).astype('float32')
faiss.normalize_L2(vecs)  # normalize so inner product = cosine similarity

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

k = 2  # self + nearest neighbour
sims, idxs = index.search(vecs, k)

nearest_sim = sims[:, 1]          # column 0 is self (sim=1.0)
nearest_idx = idxs[:, 1]

is_duplicate = nearest_sim >= DUP_SIM_THRESHOLD
print(f"Flagged {is_duplicate.sum():,} / {len(vecs):,} tweets as near-duplicates, {100*is_duplicate.sum()/len(vecs):.1f}% at threshold {DUP_SIM_THRESHOLD}")

dup_df = pd.DataFrame({
    'tweet_id': embeddings_df['tweet_id'].values,
    'nearest_neighbor_tweet_id': embeddings_df['tweet_id'].values[nearest_idx],
    'nearest_similarity': nearest_sim,
    'is_near_duplicate': is_duplicate
})

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(dup_df, f"{PROJECT}.{TWEET_DUPLICATES}",
                                   job_config=job_config).result()
print(f"Loaded duplicate flags to {TWEET_DUPLICATES}")

Flagged 93,308 / 150,607 tweets as near-duplicates, 62.0% at threshold 0.97
Loaded duplicate flags to ngx.tweet_duplicates


## BERTopic — fit once on the full corpus**Critical:** fit exactly once here. Every window is `.transform()`-ed into this fixedtopic space downstream (stage 7b), so topic IDs stay comparable across trading days.Near-duplicates are excluded from the fit so templated spam doesn't dominate a cluster.

In [ ]:
from bertopic import BERTopicfrom sklearn.feature_extraction.text import CountVectorizerfit_mask = ~dup_df.set_index('tweet_id').loc[embeddings_df['tweet_id']]['is_near_duplicate'].valuesfit_texts = df.set_index('tweet_id').loc[embeddings_df['tweet_id'][fit_mask]]['text'].tolist()fit_vecs = vecs[fit_mask]print(f"Fitting BERTopic on {len(fit_texts):,} de-duplicated tweets...")vectorizer_model = CountVectorizer(stop_words=None, min_df=5)topic_model = BERTopic(    embedding_model=None,   # we supply precomputed embeddings    vectorizer_model=vectorizer_model,    calculate_probabilities=False,    verbose=True)t0 = time.time()topics, _ = topic_model.fit_transform(fit_texts, embeddings=fit_vecs)print(f"BERTopic fit in {(time.time()-t0)/60:.1f} minutes, "      f"{len(set(topics)) - (1 if -1 in topics else 0)} topics found")# Persist the fitted model to GCS so it can be reloaded in stage 7b for .transform()import osos.makedirs("/content/bertopic_model", exist_ok=True)topic_model.save("/content/bertopic_model/model", serialization="pickle")bucket.blob("derived/bertopic_model/model").upload_from_filename("/content/bertopic_model/model")print("BERTopic model saved to GCS at derived/bertopic_model/model")

In [ ]:
# Transform ALL tweets (including near-duplicates) into the fixed topic spaceall_texts = df.set_index('tweet_id').loc[embeddings_df['tweet_id']]['text'].tolist()all_topics, _ = topic_model.transform(all_texts, embeddings=vecs)topics_df = pd.DataFrame({    'tweet_id': embeddings_df['tweet_id'].values,    'topic_id': all_topics})job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")client.load_table_from_dataframe(topics_df, f"{PROJECT}.{TWEET_TOPICS}",                                   job_config=job_config).result()print(f"Loaded topic assignments to {TWEET_TOPICS}")

## AfriSenti — sentiment classificationProduces per-tweet bullish/neutral/bearish probabilities feeding f12 (sentiment skew).

In [ ]:
from transformers import AutoModelForSequenceClassification, pipelineSENTIMENT_MODEL = "Davlan/afrisenti-twitter-sentiment-afriberta-large"  # verify against AfriSenti repo before final runsent_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)sent_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL).to(device)sent_pipe = pipeline("text-classification", model=sent_model, tokenizer=sent_tokenizer,                      device=0 if device == "cuda" else -1, top_k=None, truncation=True,                      max_length=128, batch_size=BATCH_SIZE)t0 = time.time()results = sent_pipe(df['text'].tolist())print(f"Sentiment scored in {(time.time()-t0)/60:.1f} minutes")# Flatten to a bullish-probability column (adjust label names to match model's actual output)def bullish_prob(scores):    d = {s['label'].lower(): s['score'] for s in scores}    return d.get('positive', d.get('bullish', 0.0))sent_df = pd.DataFrame({    'tweet_id': df['tweet_id'].values,    'bullish_prob': [bullish_prob(r) for r in results]})job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")client.load_table_from_dataframe(sent_df, f"{PROJECT}.{TWEET_SENTIMENT}",                                   job_config=job_config).result()print(f"Loaded sentiment scores to {TWEET_SENTIMENT}")

## VerificationConfirms row counts line up before moving to stage 7b (feature computation).

In [ ]:
print("Tweets embedded:      ", len(embeddings_df))print("Duplicate flags:      ", len(dup_df), f"({dup_df['is_near_duplicate'].mean()*100:.1f}% flagged)")print("Topic assignments:    ", len(topics_df))print("Sentiment scores:     ", len(sent_df))print("Source tweets (input):", len(df))